Loading dataframe from file 

In [ ]:
import pandas as pd
df=pd.read_csv('../Data/Raw/spam.csv', encoding='latin-1')
print(df.shape)
df=df[['v1','v2']]
df.columns=['label','text']
print(df.head())

(Eda)Checking the Class balance 

In [ ]:
df.label.value_counts()

Checking for duplicates and missing values 

In [ ]:
print(df.duplicated().sum())
print(df.isnull().sum())

Creating a new cleaned dataframe (without duplicates)
Reason : duplicates in training data will creat an overfitting problem for my model and will cause a dataleakage if they are present in testing data  

In [ ]:
df =df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(df.duplicated().sum())

Checking the intergrity of the data types 

In [ ]:
print(df.info())

Adding a length column to help me indentify spam and ham messages 

In [ ]:
df['length']=df['text'].apply(len)
print(df.groupby('label')['length'].describe())

histogram

In [ ]:
df[df.label=='spam']['length'].hist()

In [ ]:
df[df.label=='ham']['length'].hist()

## Text Preprocessing
### 1. Cleaning 
**Operation** Removing uppercase letters, numbers and punctuation (!,.?)

In [ ]:
df['text']=df['text'].str.lower()
df['text']=df['text'].str.replace(r'[0-9\W_]+', ' ', regex=True)

### 2. Tokenization 
**Operation** Removing stop_words (is,the...) and stemming the words (playing into play)
**Goal** This process simplifies the data (removing noise and keeping actual useful data) so that we can work on it

In [ ]:
import nltk # this is an nlp toolkit
from nltk.corpus import stopwords # this is an object of a list of common words we can remove
from nltk.stem import PorterStemmer # this is the class of the objet that reduces words to their root form
nltk.download('stopwords') # the actual list

stemmer=PorterStemmer()
stop_words=set(stopwords.words('english'))
l_tokens=df['text'].str.split()
nl_tokens=[]
for tokens in l_tokens:
    tokens=[stemmer.stem(token) for token in tokens if token not in stop_words]
    nl_tokens.append(' '.join(tokens))
df['cleaned_text']=nl_tokens
#Comparing the original text and the cleaned text
print(  df['text'].head(),"\n")
print(df['cleaned_text'].head())

## Feature Extraction (turning text into numbers)
### Tf-Idf vectorizer 
**Goal** Transforming text data into numerical one and exploring it

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=3000)
X=vectorizer.fit_transform(df['cleaned_text'])
print(X)



## Saving the cleaned_Data into a csv file 

In [ ]:
df.to_csv('../Data/Processed/spam_cleaned.csv', index=False)